# 策略研究

算法写在项目源码的 `algo.py`，运行参数定义在 `ModelParams`。表单的 `build()` 构造继承它的本项目 `ModelAnalysisParams`，再通过 `run(ModelAlgo)` 运行研究。示例在收盘后选出股票池内近 20 个交易日收益最高的股票，由默认后续 Algo 完成仓位计算和下单。

首个代码单元格优先加载当前项目 `src` 源码；修改源码后重跑此格，不使用环境中旧安装包。请在当前项目目录运行内核。

In [ ]:
from pathlib import Path
import importlib
import sys
import tomllib

project_root = Path.cwd().resolve()
pyproject_path = project_root / "pyproject.toml"
if not pyproject_path.is_file():
    raise FileNotFoundError("请将内核工作目录设为包含 pyproject.toml 和 src 的当前项目目录。")
with pyproject_path.open("rb") as project_file:
    project_config = tomllib.load(project_file)
package_name = (
    project_config.get("tool", {}).get("uv", {}).get("build-backend", {}).get("module-name")
    or project_config.get("project", {}).get("name", "").replace("-", "_")
)
src_path = (project_root / "src").resolve()
if not package_name or not src_path.is_dir():
    raise FileNotFoundError("当前项目缺少包名配置或 src 目录；请检查 pyproject.toml 和工作目录。")
package_path = src_path.joinpath(*package_name.split("."))
if not package_path.is_dir() and not package_path.with_suffix(".py").is_file():
    raise FileNotFoundError(f"找不到当前项目源码：{package_path}；请检查包名配置。")
sys.path[:] = [str(src_path)] + [path for path in sys.path if path != str(src_path)]
importlib.invalidate_caches()
for loaded_name in list(sys.modules):
    if loaded_name == package_name or loaded_name.startswith(package_name + "."):
        del sys.modules[loaded_name]

from model import ModelAlgo, ModelAnalysisParams, ModelReportForm
from scheme import StockPool

In [ ]:
form = ModelReportForm(
    start="2026-06-01",
    end="2026-06-06",
    pool=StockPool.SSE50,
    benchmark="000300.SH",
    cash=1_000_000,
    n_select=10,
    optimize="risk_parity",
    control="no_control",
    execution="direct_execution",
)
params: ModelAnalysisParams = form.build()
params

In [ ]:
report = params.run(ModelAlgo)
report.show()